<a href="https://colab.research.google.com/github/jarl24-dev/mlops-zoomcamp/blob/main/02-dataframe-analysis/Homework2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Question 1: [IPO] Withdrawn IPOs by Company Type

In [108]:
import pandas as pd
import requests
from io import StringIO

In [109]:
url='https://stockanalysis.com/ipos/withdrawn/'
headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/58.0.3029.110 Safari/537.3'
        )
    }

response = requests.get(url, headers=headers, timeout=10)
html_io = StringIO(response.text)

tables = pd.read_html(html_io)
df = tables[0]

In [110]:
df["Company Name"] = df["Company Name"].astype("string")

def companyclass(company_name):
    if "Acquisition Corp" in company_name or "Acquisition Corporation" in company_name:
        return "Acq.Corp"
    elif "Inc" in company_name or "Incorporated" in company_name:
        return "Inc"
    elif "Group" in company_name:
        return "Group"
    elif "Ltd" in company_name or "Limited" in company_name:
        return "Limited"
    elif "Holdings" in company_name:
        return "Holdings"
    else:
        return "Other"

df["Company Class"] = df["Company Name"].apply(companyclass)
df["Company Class"] = df["Company Class"].astype("string")

In [111]:
df["Price Range"] = df["Price Range"].astype("string")

def avgprice(price_range):
  price_range = price_range.split("-")
  for i in range(len(price_range)):
    try:
      price_range[i]=float(price_range[i].replace("$", "").replace(" ", ""))
    except:
      price_range[i]=0.0
  return sum(price for price in price_range) / len(price_range)

df["Avg. price"] = df["Price Range"].apply(avgprice)
df["Avg. price"] = df["Avg. price"].astype(float)

In [112]:
df["Shares Offered"] = pd.to_numeric(df["Shares Offered"], errors="coerce")

In [113]:
df["Withdrawn Value"] = df["Shares Offered"] * df["Avg. price"]

In [114]:
df.groupby("Company Class")["Withdrawn Value"].sum().sort_values(ascending=False)

,Withdrawn Value
Company Class,
Acq.Corp,4.021000e+09
Inc,2.257164e+09
Other,7.679200e+08
Limited,5.497346e+08
Holdings,7.500000e+07
Group,3.378750e+07


# Question 2: [IPO] Median Sharpe Ratio for 2024 IPOs (First 5 Months)

In [139]:
import pandas as pd
import requests
from io import StringIO

In [140]:
url='https://stockanalysis.com/ipos/2024/'
headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/58.0.3029.110 Safari/537.3'
        )
    }

response = requests.get(url, headers=headers, timeout=10)
html_io = StringIO(response.text)

tables = pd.read_html(html_io)
df = tables[0]

df['IPO Date'] = pd.to_datetime(df['IPO Date'], errors="coerce")
df_new = df[(df['IPO Date'] < "2024-06-01") & (df['IPO Price']!='-')]

In [142]:
ALL_TICKERS = df_new['Symbol'].values

In [146]:
import time
import yfinance as yf
import numpy as np

stocks_df = pd.DataFrame({'A' : []})

for i,ticker in enumerate(ALL_TICKERS):
  print(i,ticker)

  # Work with stock prices
  ticker_obj = yf.Ticker(ticker)

  # historyPrices = yf.download(tickers = ticker,
  #                    period = "max",
  #                    interval = "1d")
  historyPrices = ticker_obj.history(
                     period = "max",
                     interval = "1d")

  # generate features for historical prices, and what we want to predict
  historyPrices['Ticker'] = ticker
  historyPrices['Year']= historyPrices.index.year
  historyPrices['Month'] = historyPrices.index.month
  historyPrices['Weekday'] = historyPrices.index.weekday
  historyPrices['Date'] = historyPrices.index.date

  # historical returns
  for i in [1,3,7,30,90,252,365]:
    historyPrices['growth_'+str(i)+'d'] = historyPrices['Close'] / historyPrices['Close'].shift(i)
  historyPrices['growth_future_30d'] = historyPrices['Close'].shift(-30) / historyPrices['Close']

  # Technical indicators
  # SimpleMovingAverage 10 days and 20 days
  historyPrices['SMA10']= historyPrices['Close'].rolling(10).mean()
  historyPrices['SMA20']= historyPrices['Close'].rolling(20).mean()
  historyPrices['growing_moving_average'] = np.where(historyPrices['SMA10'] > historyPrices['SMA20'], 1, 0)
  historyPrices['high_minus_low_relative'] = (historyPrices.High - historyPrices.Low) / historyPrices['Close']

  # 30d rolling volatility : https://ycharts.com/glossary/terms/rolling_vol_30
  historyPrices['volatility'] =   historyPrices['Close'].rolling(30).std() * np.sqrt(252)

  # what we want to predict
  historyPrices['is_positive_growth_30d_future'] = np.where(historyPrices['growth_future_30d'] > 1, 1, 0)

  # sleep 1 sec between downloads - not to overload the API server
  time.sleep(1)


  if stocks_df.empty:
    stocks_df = historyPrices
  else:
    stocks_df = pd.concat([stocks_df, historyPrices], ignore_index=True)

0 BOW
1 HDL
2 RFAI
3 JDZG
4 RAY
5 BTOC
6 ZK
7 GPAT
8 PAL
9 SVCO
10 NNE
11 CCIX
12 VIK
13 ZONE
14 LOAR
15 MRX
16 RBRK
17 NCI
18 MFI
19 YYGH
20 TRSG
21 CDTG
22 CTRI
23 IBTA
24 MTEN
25 TWG
26 ULS
27 PACS
28 MNDR
29 CTNM
30 MAMO
31 ZBAO
32 BOLD
33 MMA
34 UBXG
35 IBAC
36 AUNA
37 BKHA
38 LOBO
39 RDDT
40 ALAB
41 INTJ
42 RYDE
43 LGCL
44 SMXT
45 VHAI
46 DYCQ
47 CHRO
48 UMAC
49 HLXB
50 MGX
51 TBBB
52 TELO
53 KYTX
54 PMNT
55 AHR
56 LEGT
57 ANRO
58 GUTS
59 AS
60 FBLG
61 AVBP
62 BTSG
63 HAO
64 CGON
65 YIBO
66 JL
67 SUGP
68 JVSA
69 KSPI
70 CCTG
71 PSBD
72 SYNX
73 SDHC
74 ROMA


In [147]:
stocks_df['Sharpe'] = (stocks_df['growth_252d'] - 0.045) / stocks_df['volatility']

In [155]:
stocks_df['Date'] = pd.to_datetime(stocks_df['Date'], errors="coerce")
stocks_filtered = stocks_df[stocks_df['Date'] == '2025-06-06']

In [162]:
stocks_filtered[['growth_252d','Sharpe']].describe()

,growth_252d,Sharpe
count,71.000000,71.000000
mean,1.152897,0.288285
std,1.406017,0.519028
min,0.024970,-0.079677
25%,0.293422,0.041215
50%,0.758065,0.083768
75%,1.362736,0.311507
max,8.097413,2.835668
